In [4]:
from pathlib import Path
from typing import Union
import pandas as pd


#This debug implementation is meant to create a table that allows us to map the predictions
#to timestamps. This will allow me to see exactly what the prediction is for and lets me 
#visually cross check on a chart. The prediction data from weka has to be manually pasted 
#into weka_predictions.txt for it to work properly.


def parse_weka_predictions(prediction_text_path: Union[str, Path]) -> pd.DataFrame:
    
    """
    Parses WEKA plain-text prediction output.

    Expected WEKA format:
        inst# actual predicted error prediction
        18 1:not_range 2:range + 1
        26 1:not_range 1:not_range 1

    Returns:
        DataFrame with:
            weka_inst
            actual
            predicted
            is_error
            prediction_confidence
    """

    prediction_text_path = Path(prediction_text_path)

    if not prediction_text_path.exists():
        
        raise FileNotFoundError(f"Prediction text file does not exist: {prediction_text_path}")


    rows = []
    

    with prediction_text_path.open("r", encoding="utf-8") as f:
        
        for line in f:
            
            line = line.strip()

            #no headers/blank lines
            if not line or line.startswith("inst#") or line.startswith("==="):
                continue

            parts = line.split()

            #prediction rows start with instance number
            if not parts[0].isdigit():
                continue

            weka_inst = int(parts[0])
            actual_raw = parts[1]
            predicted_raw = parts[2]

            actual = actual_raw.split(":", 1)[1]
            predicted = predicted_raw.split(":", 1)[1]

            #errors on weka predictions appear as "+"
            is_error = "+" in parts

            #last numeric item is usually prediction/confidence
            prediction_confidence = None
            
            for item in reversed(parts):
                
                try:
                    prediction_confidence = float(item)
                    break
                
                except ValueError:
                    pass

            
            rows.append({
                "weka_inst": weka_inst,
                "actual": actual,
                "predicted": predicted,
                "is_error": is_error,
                "prediction_confidence": prediction_confidence
            })

    
    predictions_df = pd.DataFrame(rows)

    
    if predictions_df.empty:
        
        raise ValueError("No WEKA prediction rows were parsed. Check the prediction text format.")


    
    return predictions_df

    


def map_weka_predictions_to_full_features(*,
                                          prediction_text_path: Union[str, Path] = "feature_data/weka/weka_predictions.txt",
                                          debug_test_path: Union[str, Path] = "feature_data/weka/EURUSD_D1_range_weka_test_debug.csv",
                                          full_feature_path: Union[str, Path] = "feature_data/range_training_features/EURUSD_D1_range_full_features.parquet",
                                          output_dir: Union[str, Path] = "feature_data/weka/prediction_review"
                                         ) -> tuple[pd.DataFrame, pd.DataFrame]:
    
    """
    Maps WEKA test predictions back to full feature rows. Main purpose is to manually see results visually on a chart
    as mentioned in header.

    Saves:
        all_predictions_mapped.csv
        mistake_predictions_mapped.csv

    Returns:
        all_mapped_predictions, mistake_predictions
    """

    prediction_text_path = Path(prediction_text_path)
    debug_test_path = Path(debug_test_path)
    full_feature_path = Path(full_feature_path)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    predictions_df = parse_weka_predictions(prediction_text_path)

    debug_test_df = pd.read_csv(debug_test_path)
    full_features = pd.read_parquet(full_feature_path)

    
    if "row_id" not in debug_test_df.columns:
        
        raise ValueError("debug_test_df must contain a row_id column.")

    
    #weka inst# starts at 1
    predictions_df["debug_test_position"] = predictions_df["weka_inst"] - 1

    
    #insert row_id from debug test file
    debug_lookup = debug_test_df.reset_index(drop=True).copy()
    debug_lookup["debug_test_position"] = debug_lookup.index

    
    mapped = predictions_df.merge(
        debug_lookup,
        on="debug_test_position",
        how="left",
        suffixes=("", "_debug")
    )

    
    if mapped["row_id"].isna().any():
        
        bad_rows = mapped[mapped["row_id"].isna()]
        
        raise ValueError("Some WEKA instances could not be mapped to debug_test_df. "
                         f"Problem rows:\n{bad_rows}")

    
    mapped["row_id"] = mapped["row_id"].astype(int)

    #pull matching full feature rows using row_id as original iloc position
    full_rows = full_features.iloc[mapped["row_id"].values].reset_index(drop=True).copy()

    #prefix full feature columns to avoid name collisions
    full_rows = full_rows.add_prefix("full_")

    all_mapped = pd.concat([mapped.reset_index(drop=True),full_rows], axis=1)

    mistake_rows = all_mapped[all_mapped["is_error"]].copy()

    all_output_path = output_dir / "all_predictions_mapped.csv"
    mistake_output_path = output_dir / "mistake_predictions_mapped.csv"

    all_mapped.to_csv(all_output_path, index=False)
    mistake_rows.to_csv(mistake_output_path, index=False)

    print("[PREDICTION MAP] Saved all mapped predictions to:", all_output_path)
    print("[PREDICTION MAP] Saved mistake predictions to:", mistake_output_path)
    print()
    print("[PREDICTION MAP] Total predictions:", len(all_mapped))
    print("[PREDICTION MAP] Mistakes:", len(mistake_rows))
    print()
    print("[PREDICTION MAP] Prediction counts:")
    print(all_mapped[["actual", "predicted"]].value_counts())
    print()
    print("[PREDICTION MAP] Mistake counts:")
    
    if len(mistake_rows) > 0:
        
        print(mistake_rows[["actual", "predicted"]].value_counts())
    
    else:
        
        print("No mistakes found.")

    
    return all_mapped, mistake_rows


all_predictions, mistakes = map_weka_predictions_to_full_features()

[PREDICTION MAP] Saved all mapped predictions to: feature_data/weka/prediction_review/all_predictions_mapped.csv
[PREDICTION MAP] Saved mistake predictions to: feature_data/weka/prediction_review/mistake_predictions_mapped.csv

[PREDICTION MAP] Total predictions: 289
[PREDICTION MAP] Mistakes: 11

[PREDICTION MAP] Prediction counts:
actual     predicted
range      range        145
not_range  not_range    133
           range         11
Name: count, dtype: int64

[PREDICTION MAP] Mistake counts:
actual     predicted
not_range  range        11
Name: count, dtype: int64
